# TEST BE CAREFUL !

In [ ]:
# Finalize disease table for downstream merges
# Build a condensed diseases dataframe with key fields
final_diseases_df = pd.DataFrame({
    "disease_id": diseases_df["id"],
    "disease_name": diseases_df["name"].str.lower(),
    "uniprot_ids": diseases_df.get("uniprot_ids", [[]]),
    "phenotypes": diseases_df.get("phenotypes", [[]]),
})
final_diseases_df["num_targets"] = final_diseases_df["uniprot_ids"].apply(lambda xs: len(xs) if isinstance(xs, list) else 0)

# Ensure unique diseases
final_diseases_df = (
    final_diseases_df
    .drop_duplicates(subset=["disease_id"])  # safety
    .reset_index(drop=True)
)

# Light sanity: keep only diseases present in embeddings_df (if computed)
if 'embeddings_df' in globals():
    final_diseases_df = final_diseases_df[
        final_diseases_df["disease_id"].isin(embeddings_df["disease_id"].unique())
    ].reset_index(drop=True)



In [ ]:
# Build pairwise dataframe: one row per (drug_id, disease_id)
# Uses embeddings_df as the base (already a full cross-product with similarity)

def _aggregate_drug_uniprot_set(targets_dict):
    if not isinstance(targets_dict, dict):
        return set()
    agg = set()
    for action, ids in targets_dict.items():
        if isinstance(ids, list):
            agg.update([i for i in ids if isinstance(i, str)])
    return agg

# Prepare drug-side features
_drug_cols_keep = ["drug_id", "drug_name", "targets"]
drug_side = final_drugs_df[_drug_cols_keep].copy()
drug_side["drug_uniprot_set"] = drug_side["targets"].apply(_aggregate_drug_uniprot_set)

# Prepare disease-side features
_disease_cols_keep = ["disease_id", "disease_name", "uniprot_ids", "num_targets"]
disease_side = final_diseases_df[_disease_cols_keep].copy()
disease_side.rename(columns={"uniprot_ids": "disease_uniprot_list"}, inplace=True)
disease_side["disease_uniprot_set"] = disease_side["disease_uniprot_list"].apply(lambda xs: set(xs) if isinstance(xs, list) else set())

# Base (drug, disease) from embeddings
base_pairs = embeddings_df[["drug_id", "disease_id", "name_similarity"]].copy()

# Merge drug/disease features
pairwise_df = base_pairs.merge(drug_side, on="drug_id", how="left").merge(
    disease_side, on="disease_id", how="left"
)

# Add indication evidence (ChEMBL + NCT summary)
if 'final_indications_df' in globals():
    ind_keep = ["drug_id", "efo_id", "max_phase_for_ind", "nct_evidence"]
    ind_side = final_indications_df[ind_keep].copy()
    ind_side.rename(columns={"efo_id": "disease_id"}, inplace=True)
    ind_side["has_known_indication"] = True
    pairwise_df = pairwise_df.merge(ind_side, on=["drug_id", "disease_id"], how="left")
else:
    pairwise_df["max_phase_for_ind"], pairwise_df["nct_evidence"], pairwise_df["has_known_indication"] = (None, None, False)

# Compute target overlaps
pairwise_df["shared_targets_count"] = pairwise_df.apply(
    lambda r: len(r["drug_uniprot_set"] & r["disease_uniprot_set"]) if isinstance(r.get("drug_uniprot_set"), set) and isinstance(r.get("disease_uniprot_set"), set) else 0,
    axis=1,
)
pairwise_df["targets_jaccard"] = pairwise_df.apply(
    lambda r: (
        (len(r["drug_uniprot_set"] & r["disease_uniprot_set"]) / len(r["drug_uniprot_set"] | r["disease_uniprot_set"]))
        if isinstance(r.get("drug_uniprot_set"), set) and isinstance(r.get("disease_uniprot_set"), set) and len(r["drug_uniprot_set"] | r["disease_uniprot_set"]) > 0
        else 0.0
    ),
    axis=1,
)

# Basic ordering for convenience
first_cols = [
    "drug_id", "drug_name", "disease_id", "disease_name",
    "name_similarity", "has_known_indication", "max_phase_for_ind",
    "shared_targets_count", "targets_jaccard",
]
pairwise_df = pairwise_df[first_cols + [c for c in pairwise_df.columns if c not in first_cols]]



In [ ]:
# Export NetworkX graph of Drugs–Targets–Diseases with attributes
import os
import networkx as nx

# Ensure results directory exists
os.makedirs("../results", exist_ok=True)

G = nx.DiGraph()

# Add disease nodes
for _, row in final_diseases_df.iterrows():
    G.add_node(
        f"disease::{row['disease_id']}",
        kind="disease",
        disease_id=row["disease_id"],
        name=row["disease_name"],
        num_targets=row["num_targets"],
    )

# Collect all target UniProt IDs from drugs and diseases
all_target_ids = set()
# From diseases
for ids in final_diseases_df["uniprot_ids"]:
    if isinstance(ids, list):
        all_target_ids.update([i for i in ids if isinstance(i, str)])
# From drugs
if "final_drugs_df" in globals():
    for _, drow in final_drugs_df.iterrows():
        tdict = drow.get("targets")
        if isinstance(tdict, dict):
            for _, ids in tdict.items():
                if isinstance(ids, list):
                    all_target_ids.update([i for i in ids if isinstance(i, str)])

# Add target nodes
for up_id in all_target_ids:
    G.add_node(f"target::{up_id}", kind="target", uniprot=up_id)

# Add drug nodes
for _, drow in final_drugs_df.iterrows():
    G.add_node(
        f"drug::{drow['drug_id']}",
        kind="drug",
        drug_id=drow["drug_id"],
        name=drow["drug_name"],
    )

# Add drug–target edges with action_type where available
for _, drow in final_drugs_df.iterrows():
    tdict = drow.get("targets")
    if not isinstance(tdict, dict):
        continue
    for action_type, ids in tdict.items():
        if not isinstance(ids, list):
            continue
        for up_id in ids:
            if isinstance(up_id, str):
                G.add_edge(
                    f"drug::{drow['drug_id']}",
                    f"target::{up_id}",
                    kind="drug-target",
                    action_type=action_type,
                )

# Add target–disease edges (association from Open Targets)
for _, row in final_diseases_df.iterrows():
    for up_id in (row.get("uniprot_ids") or []):
        if isinstance(up_id, str):
            G.add_edge(
                f"target::{up_id}",
                f"disease::{row['disease_id']}",
                kind="target-disease",
            )

# Add drug–disease edges for known indications with attrs
if 'final_indications_df' in globals():
    for _, irow in final_indications_df.iterrows():
        src = f"drug::{irow['drug_id']}"
        dst = f"disease::{irow['efo_id']}"
        G.add_edge(
            src,
            dst,
            kind="drug-disease",
            source="ChEMBL",
            max_phase_for_ind=irow.get("max_phase_for_ind"),
            nct_evidence=irow.get("nct_evidence"),
        )

# Optionally, add similarity as a weighted edge for all pairs (sparse example: only top-k per drug)
# Here: add top 5 similar diseases per drug by name similarity
if 'pairwise_df' in globals():
    topk = (
        pairwise_df
        .sort_values(["drug_id", "name_similarity"], ascending=[True, False])
        .groupby("drug_id")
        .head(5)
    )
    for _, prow in topk.iterrows():
        G.add_edge(
            f"drug::{prow['drug_id']}",
            f"disease::{prow['disease_id']}",
            kind="drug-disease-similarity",
            weight=float(prow.get("name_similarity", 0.0)),
            feature="name_similarity",
        )

# Save artifacts
pairwise_path_parquet = "../results/pairwise.parquet"
pairwise_path_csv = "../results/pairwise.csv"
if 'pairwise_df' in globals():
    try:
        pairwise_df.to_parquet(pairwise_path_parquet, index=False)
    except Exception:
        pass
    pairwise_df.to_csv(pairwise_path_csv, index=False)

# Graph exports
nx.write_graphml(G, "../results/repurposing.graphml")
# Optional lighter exports
nx.write_gpickle(G, "../results/repurposing.gpickle")

